In [1]:
# =============================================================================
# CISLR — STAGE 5
# Official prototype -> test retrieval evaluation.
# =============================================================================
#
# What this cell does:
#   1.  Locates and loads the existing EfficientNet-B0 feature cache.
#   2.  Loads prototype.csv and test.csv, enforces the test boundary.
#   3.  Loads the Stage-6.2 temporal checkpoints (frozen) and constructs
#       retrieval embeddings for each of the four models.
#   4.  Computes cosine-similarity retrieval of test videos against the
#       prototype gallery (4763 prototypes).
#   5.  Reports Top-1, Top-5, Top-10, MRR, per-class Top-1, and per-class
#       breakdown by test-example count.
#   6.  Saves all artifacts.
#   7.  Stops BEFORE Stage 8.
#
# No training, no gradient updates, no test-label usage for anything except
# the final metric computation.
# =============================================================================

import os, json, time, math, random, warnings, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

ROOT = Path("/home/dipshikha/Music/cao")
DATASET_CSV   = ROOT / "dataset.csv"
PROTOTYPE_CSV = ROOT / "prototype.csv"
TEST_CSV      = ROOT / "test.csv"
RESEARCH_ROOT = ROOT / "CISLR_RESEARCH"

STAGE6_2_DIR = RESEARCH_ROOT / "audit" / "stage6_2_temporal_representation"
OUT_DIR      = RESEARCH_ROOT / "audit" / "stage7_official_retrieval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_DIM = 1280
NUM_FRAMES  = 8
HIDDEN_DIM  = 256
DROPOUT     = 0.2

BATCH_SIZE = 64
TIE_REL_TOL = 0.01

MISSING_LABEL_UID = "R_GaklgD4bY=-"

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

# Known Mean Pool baseline from an earlier valid evaluation
KNOWN_MEAN_POOL = {"top1": 0.1274, "top5": 0.2236, "top10": 0.2600, "mrr": 0.1702}

print("=" * 90)
print("STAGE 7 — OFFICIAL CISLR PROTOTYPE → TEST RETRIEVAL")
print("=" * 90)
print(f"Device           : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
print(f"PyTorch          : {torch.__version__}")
print(f"Output directory : {OUT_DIR}")


# =============================================================================
# 2. LOAD METADATA AND ENFORCE TEST BOUNDARY
# =============================================================================

print("\n" + "=" * 90)
print("LOADING METADATA")
print("=" * 90)

def normalize_uid(x):
    if pd.isna(x): return None
    return str(x).strip()

def normalize_gloss(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}: return None
    return x

prototype = pd.read_csv(PROTOTYPE_CSV)
test_df   = pd.read_csv(TEST_CSV)

prototype["_uid"]   = prototype["uid"].apply(normalize_uid)
prototype["_gloss"] = prototype["gloss"].apply(normalize_gloss)
test_df["_uid"]     = test_df["uid"].apply(normalize_uid)
test_df["_gloss"]   = test_df["gloss"].apply(normalize_gloss)

print(f"prototype.csv : {len(prototype):,} rows")
print(f"test.csv      : {len(test_df):,} rows")

# ---- Missing-label exclusion ----
missing_label_present = MISSING_LABEL_UID in set(prototype["_uid"].dropna())
print(f"Missing-label UID '{MISSING_LABEL_UID}' present in prototype: {missing_label_present}")

proto_valid = prototype.dropna(subset=["_uid", "_gloss"])
proto_valid = proto_valid[proto_valid["_uid"] != MISSING_LABEL_UID].copy()

test_valid = test_df.dropna(subset=["_uid", "_gloss"]).copy()

print(f"Prototype rows with valid gloss (after missing-label exclusion): {len(proto_valid):,}")
print(f"Test rows with valid gloss: {len(test_valid):,}")

# Duplicate checks
assert proto_valid["_uid"].is_unique, "Duplicate prototype UIDs present."
assert test_valid["_uid"].is_unique, "Duplicate test UIDs present."
print("PASS | no duplicate prototype UIDs")
print("PASS | no duplicate test UIDs")

proto_uids_set = set(proto_valid["_uid"])
test_uids_set  = set(test_valid["_uid"])

# Disjointness
assert len(proto_uids_set & test_uids_set) == 0, \
    "Prototype and test UIDs overlap."
print("PASS | prototype UIDs ∩ test UIDs = empty")

# Every test class has exactly one prototype
proto_class_to_uid = proto_valid.set_index("_gloss")["_uid"].to_dict()
missing_proto = set(test_valid["_gloss"]) - set(proto_class_to_uid.keys())
assert len(missing_proto) == 0, \
    f"Test classes without prototype: {sorted(missing_proto)[:10]}"
print("PASS | every test class has exactly one prototype")

# Report test UID count
print(f"Test UID count reported by spec: 2285")
print(f"Test UID count observed        : {len(test_uids_set):,}")


# =============================================================================
# 3. LOCATE AND LOAD FEATURE CACHE
# =============================================================================

print("\n" + "=" * 90)
print("LOCATING FEATURE CACHE")
print("=" * 90)

FEATURE_DIR = RESEARCH_ROOT / "models" / "baseline" / "efficientnet_temporal_retrieval" / "features"
search_roots = [FEATURE_DIR, RESEARCH_ROOT]
cache_candidates = []
for root in search_roots:
    if not root.exists(): continue
    for pattern in ("all_video_features_*.pt",
                    "all_video_efficientnet_b0_features*.pt",
                    "all_video*.pt"):
        cache_candidates.extend(sorted(root.rglob(pattern)))

seen = set(); uniq = []
for c in cache_candidates:
    if c.resolve() in seen: continue
    seen.add(c.resolve()); uniq.append(c)

if not uniq:
    raise FileNotFoundError("No feature cache found.")

uniq.sort(key=lambda p: p.stat().st_mtime, reverse=True)
feature_cache_path = uniq[0]
print(f"Using cache: {feature_cache_path}")
print(f"  (found {len(uniq)} candidates; using newest)")

feature_cache = torch.load(feature_cache_path, map_location="cpu")
print(f"Cached videos: {len(feature_cache):,}")

sample_key = next(iter(feature_cache))
sample_shape = tuple(feature_cache[sample_key].shape)
print(f"Feature shape: {sample_key} -> {sample_shape}")
assert sample_shape == (NUM_FRAMES, FEATURE_DIM), \
    f"Unexpected feature shape {sample_shape}"

# Excluded UIDs
proto_available = [u for u in proto_valid["_uid"] if u in feature_cache]
test_available  = [u for u in test_valid["_uid"]  if u in feature_cache]
proto_missing   = [u for u in proto_valid["_uid"] if u not in feature_cache]
test_missing    = [u for u in test_valid["_uid"]  if u not in feature_cache]

print(f"Prototype UIDs with cached features: {len(proto_available):,}"
      f"  (missing {len(proto_missing)})")
print(f"Test UIDs with cached features     : {len(test_available):,}"
      f"  (missing {len(test_missing)})")
if proto_missing:
    print(f"  Excluded prototype UIDs (examples): {proto_missing[:5]}")
if test_missing:
    print(f"  Excluded test UIDs (examples)     : {test_missing[:5]}")


# =============================================================================
# 4. REBUILD STAGE-6.2 ARCHITECTURES (identical definitions)
# =============================================================================

class BiLSTMRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 layers=2, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.LayerNorm(hidden))
        self.mask_token = nn.Parameter(torch.zeros(hidden))
        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim))

    def forward(self, x, mask_idx):
        B, T, D = x.shape
        h = self.input_proj(x)
        mask_onehot = F.one_hot(mask_idx, num_classes=T).float()
        h = h * (1 - mask_onehot.unsqueeze(-1)) + \
            self.mask_token.view(1, 1, -1) * mask_onehot.unsqueeze(-1)
        out, _ = self.lstm(h)
        idx = mask_idx.view(B, 1, 1).expand(-1, 1, out.shape[-1])
        masked_out = out.gather(1, idx).squeeze(1)
        return self.head(masked_out)

    def encode(self, x):
        """
        Retrieval embedding: temporal mean of BiLSTM hidden states.
        Shape: (B, 2*hidden).
        Rationale: encoder-level representation, independent of the mask
        token and the reconstruction head, so the same embedding is
        produced for every frame ordering.
        """
        h = self.input_proj(x)
        out, _ = self.lstm(h)
        return out.mean(dim=1)


class BiLSTMAttentionRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 layers=2, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.LayerNorm(hidden))
        self.mask_token = nn.Parameter(torch.zeros(hidden))
        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.attn_score = nn.Sequential(
            nn.Linear(hidden * 4, hidden), nn.Tanh(),
            nn.Linear(hidden, 1))
        self.head = nn.Sequential(
            nn.Linear(hidden * 4, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim))

    def forward(self, x, mask_idx):
        B, T, D = x.shape
        h = self.input_proj(x)
        mask_onehot = F.one_hot(mask_idx, num_classes=T).float()
        h = h * (1 - mask_onehot.unsqueeze(-1)) + \
            self.mask_token.view(1, 1, -1) * mask_onehot.unsqueeze(-1)
        out, _ = self.lstm(h)
        idx = mask_idx.view(B, 1, 1).expand(-1, 1, out.shape[-1])
        masked_out = out.gather(1, idx).squeeze(1)
        masked_exp = masked_out.unsqueeze(1).expand(-1, T, -1)
        pair = torch.cat([masked_exp, out], dim=-1)
        scores = self.attn_score(pair).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = (weights * out).sum(dim=1)
        combined = torch.cat([masked_out, context], dim=-1)
        return self.head(combined)

    def encode(self, x):
        """
        Retrieval embedding: attention-weighted sum of BiLSTM hidden states.
        Shape: (B, 2*hidden).
        The attention weights are computed with a learned query that is
        independent of any masked position, so the embedding is a pure
        function of the sequence.
        """
        B, T, D = x.shape
        h = self.input_proj(x)
        out, _ = self.lstm(h)                       # (B, T, 2H)
        # Learned query: use the mean of out as the query (data-independent
        # attention target), then compute weights.
        query = out.mean(dim=1, keepdim=True).expand(-1, T, -1)   # (B, T, 2H)
        pair = torch.cat([query, out], dim=-1)                    # (B, T, 4H)
        scores = self.attn_score(pair).squeeze(-1)                # (B, T)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)      # (B, T, 1)
        return (weights * out).sum(dim=1)                         # (B, 2H)


class TemporalTransformerRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, d_model=HIDDEN_DIM,
                 heads=8, layers=2, ff=512, dropout=DROPOUT,
                 out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed  = nn.Parameter(torch.zeros(1, NUM_FRAMES, d_model))
        self.mask_token = nn.Parameter(torch.zeros(d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=ff,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, out_dim))

    def forward(self, x, mask_idx):
        B, T, D = x.shape
        h = self.input_proj(x) + self.pos_embed
        mask_onehot = F.one_hot(mask_idx, num_classes=T).float()
        h = h * (1 - mask_onehot.unsqueeze(-1)) + \
            self.mask_token.view(1, 1, -1) * mask_onehot.unsqueeze(-1)
        out = self.encoder(h)
        out = self.norm(out)
        idx = mask_idx.view(B, 1, 1).expand(-1, 1, out.shape[-1])
        masked_out = out.gather(1, idx).squeeze(1)
        return self.head(masked_out)

    def encode(self, x):
        """
        Retrieval embedding: temporal mean of Transformer encoder outputs.
        Shape: (B, d_model).
        """
        h = self.input_proj(x) + self.pos_embed
        out = self.encoder(h)
        out = self.norm(out)
        return out.mean(dim=1)


# =============================================================================
# 5. LOAD STAGE-6.2 CHECKPOINTS (frozen, for inference only)
# =============================================================================

print("\n" + "=" * 90)
print("LOADING STAGE-6.2 CHECKPOINTS")
print("=" * 90)

CHECKPOINTS = {
    "bilstm":               STAGE6_2_DIR / "bilstm_best.pt",
    "bilstm_attention":     STAGE6_2_DIR / "bilstm_attention_best.pt",
    "temporal_transformer": STAGE6_2_DIR / "temporal_transformer_best.pt",
}
ARCH_BUILDERS = {
    "bilstm":               BiLSTMRecon,
    "bilstm_attention":     BiLSTMAttentionRecon,
    "temporal_transformer": TemporalTransformerRecon,
}

def load_ckpt_strict_encoder(model, ckpt_path, name):
    if not Path(ckpt_path).exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    state = {k[len("module."):] if k.startswith("module.") else k: v
             for k, v in state.items()}

    model_state = model.state_dict()
    loadable = {}
    head_keys = []
    for k, v in state.items():
        if k not in model_state:
            continue
        if model_state[k].shape != v.shape:
            continue
        if k.startswith("head."):
            head_keys.append(k)
            continue  # we do not need the recon head for retrieval, but loading it is harmless
        loadable[k] = v

    missing_encoder = [k for k in model_state.keys()
                       if k not in state and not k.startswith("head.")]

    print(f"\n{name}:")
    print(f"  ckpt total keys              : {len(state):,}")
    print(f"  compatible encoder keys      : {len(loadable):,}")
    print(f"  reconstruction-head keys     : {len(head_keys):,} (loaded for completeness)")
    print(f"  missing required encoder keys: {len(missing_encoder):,}")
    if missing_encoder:
        print(f"    -> {missing_encoder[:5]}")
        raise RuntimeError(
            f"Required encoder keys missing from {name} checkpoint. Aborting "
            f"to avoid silently evaluating a partially-initialized model."
        )
    model.load_state_dict(loadable, strict=False)
    return model

loaded_models = {}
for name, path in CHECKPOINTS.items():
    model = ARCH_BUILDERS[name]()
    model = load_ckpt_strict_encoder(model, path, name)
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad = False
    loaded_models[name] = model
    print(f"  loaded for inference on {DEVICE}")


# =============================================================================
# 6. EMBEDDING EXTRACTION
# =============================================================================

print("\n" + "=" * 90)
print("EXTRACTING EMBEDDINGS")
print("=" * 90)

@torch.no_grad()
def extract_embeddings_mean_pool(uids, cache, batch_size=BATCH_SIZE):
    """Mean Pool: mean over the 8 frames. Output dim = 1280."""
    embs = []
    for i in range(0, len(uids), batch_size):
        chunk = uids[i:i+batch_size]
        x = torch.stack([cache[u].float() for u in chunk]).to(DEVICE)
        e = x.mean(dim=1)
        embs.append(e.cpu())
    return torch.cat(embs, 0)


@torch.no_grad()
def extract_embeddings_temporal(model, uids, cache, batch_size=BATCH_SIZE):
    """Use model.encode() which is the reconstruction-head-free embedding."""
    model.eval()
    embs = []
    bs = batch_size
    while True:
        try:
            for i in range(0, len(uids), bs):
                chunk = uids[i:i+bs]
                x = torch.stack([cache[u].float() for u in chunk]).to(DEVICE)
                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    e = model.encode(x)
                embs.append(e.float().cpu())
            break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            bs = bs // 2
            embs = []
            if bs < 4:
                raise RuntimeError("OOM even at batch_size=4.")
            print(f"  OOM: reducing batch size to {bs}")
    return torch.cat(embs, 0)


# UID orderings for retrieval
proto_order = [u for u in proto_valid["_uid"] if u in feature_cache]
test_order  = [u for u in test_valid["_uid"]  if u in feature_cache]
proto_gloss = [proto_valid.set_index("_uid").loc[u, "_gloss"] for u in proto_order]
test_gloss  = [test_valid.set_index("_uid").loc[u, "_gloss"]  for u in test_order]

proto_uid_to_gloss = dict(zip(proto_order, proto_gloss))
test_uid_to_gloss  = dict(zip(test_order, test_gloss))

print(f"Prototype gallery size: {len(proto_order):,}")
print(f"Test query count      : {len(test_order):,}")

# Extract for each model
model_embeddings = {}

# Mean Pool
print("\n--- Mean Pool ---")
t0 = time.time()
proto_mp = extract_embeddings_mean_pool(proto_order, feature_cache)
test_mp  = extract_embeddings_mean_pool(test_order,  feature_cache)
model_embeddings["mean_pool"] = (proto_mp, test_mp)
print(f"  prototype: {tuple(proto_mp.shape)}  test: {tuple(test_mp.shape)}  "
      f"({time.time()-t0:.1f}s)")

# Temporal models
for name, model in loaded_models.items():
    print(f"\n--- {name} ---")
    t0 = time.time()
    proto_e = extract_embeddings_temporal(model, proto_order, feature_cache)
    test_e  = extract_embeddings_temporal(model, test_order,  feature_cache)
    model_embeddings[name] = (proto_e, test_e)
    print(f"  prototype: {tuple(proto_e.shape)}  test: {tuple(test_e.shape)}  "
          f"({time.time()-t0:.1f}s)")
    torch.cuda.empty_cache()


# =============================================================================
# 7. RETRIEVAL + METRICS
# =============================================================================

def retrieval_metrics(proto_e, test_e, proto_gloss, test_gloss):
    P = F.normalize(proto_e.float(), p=2, dim=1)
    Q = F.normalize(test_e.float(),  p=2, dim=1)
    sim = Q @ P.T                              # (Q, P)
    ranks = torch.argsort(sim, dim=1, descending=True)

    top1 = top5 = top10 = 0
    rr = []
    top1_per_query = []
    for i in range(Q.shape[0]):
        order = ranks[i].tolist()
        labs  = [proto_gloss[j] for j in order]
        tgt   = test_gloss[i]
        if labs[0] == tgt: top1 += 1
        if tgt in labs[:5]: top5 += 1
        if tgt in labs[:10]: top10 += 1
        try:
            r = labs.index(tgt) + 1
        except ValueError:
            r = len(labs)
        rr.append(1.0 / r)
        top1_per_query.append(int(labs[0] == tgt))
    n = max(1, Q.shape[0])
    return {
        "top1": top1 / n, "top5": top5 / n, "top10": top10 / n,
        "mrr": float(np.mean(rr)),
        "top1_per_query": top1_per_query,
        "ranks": ranks,
        "sim": sim,
    }


print("\n" + "=" * 90)
print("RETRIEVAL")
print("=" * 90)

results = {}
for name, (proto_e, test_e) in model_embeddings.items():
    t0 = time.time()
    m = retrieval_metrics(proto_e, test_e, proto_gloss, test_gloss)
    m["embedding_dim"] = int(test_e.shape[1])
    m["elapsed"] = time.time() - t0
    results[name] = m
    print(f"  {name:24s}  Top-1 {m['top1']*100:6.2f}%  Top-5 {m['top5']*100:6.2f}%  "
          f"Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%  "
          f"({m['elapsed']:.1f}s)")


# =============================================================================
# 8. COMPARISON WITH KNOWN BASELINE
# =============================================================================

print("\n" + "=" * 90)
print("MEAN POOL BASELINE RECONCILIATION")
print("=" * 90)

recomputed_mp = results["mean_pool"]
print(f"Known   Mean Pool: Top-1 {KNOWN_MEAN_POOL['top1']*100:.2f}%  "
      f"Top-5 {KNOWN_MEAN_POOL['top5']*100:.2f}%  "
      f"Top-10 {KNOWN_MEAN_POOL['top10']*100:.2f}%  "
      f"MRR {KNOWN_MEAN_POOL['mrr']*100:.2f}%")
print(f"Current Mean Pool: Top-1 {recomputed_mp['top1']*100:.2f}%  "
      f"Top-5 {recomputed_mp['top5']*100:.2f}%  "
      f"Top-10 {recomputed_mp['top10']*100:.2f}%  "
      f"MRR {recomputed_mp['mrr']*100:.2f}%")

mp_delta = abs(recomputed_mp["top1"] - KNOWN_MEAN_POOL["top1"])
if mp_delta > 0.02:
    print(f"\nWARNING: Mean Pool Top-1 differs from the known baseline by "
          f"{mp_delta*100:.2f} percentage points.")
    print("This may be due to different feature cache versions, TTA, or the "
          "retrieval code path. All four models in this run use the SAME "
          "code path, so their relative comparison remains valid. Reconcile "
          "before drawing absolute claims.")
else:
    print(f"\nMean Pool Top-1 is within {mp_delta*100:.2f} pp of the known baseline.")


# =============================================================================
# 9. PER-CLASS ANALYSIS
# =============================================================================

print("\n" + "=" * 90)
print("PER-CLASS ANALYSIS")
print("=" * 90)

test_gloss_series = pd.Series(test_gloss)
gloss_counts = test_gloss_series.value_counts().to_dict()

per_class_rows = []
for name, m in results.items():
    top1 = m["top1_per_query"]
    by_gloss = defaultdict(list)
    for g, c in zip(test_gloss, top1):
        by_gloss[g].append(c)
    class_acc = np.array([np.mean(v) for v in by_gloss.values()])
    for g, v in by_gloss.items():
        per_class_rows.append({
            "model": name, "gloss": g, "n_test": len(v),
            "top1_frac": float(np.mean(v)),
            "n_correct": int(np.sum(v)),
        })
    print(f"  {name:24s}  classes {len(class_acc):4d}  "
          f"mean Top-1 {class_acc.mean()*100:6.2f}%  "
          f"median {np.median(class_acc)*100:6.2f}%")

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(OUT_DIR / "stage7_per_class_results.csv", index=False)


# =============================================================================
# 10. PER-EXAMPLE-COUNT BREAKDOWN
# =============================================================================

print("\n" + "=" * 90)
print("BREAKDOWN BY TEST-EXAMPLES-PER-CLASS")
print("=" * 90)

count_buckets = sorted(set(gloss_counts.values()))
breakdown_rows = []
for name, m in results.items():
    top1 = m["top1_per_query"]
    by_count = defaultdict(list)
    for g, c in zip(test_gloss, top1):
        by_count[gloss_counts[g]].append(c)
    for count in count_buckets:
        vals = by_count.get(count, [])
        if not vals: continue
        breakdown_rows.append({
            "model": name, "n_test_per_class": count,
            "n_queries": len(vals),
            "top1": float(np.mean(vals)),
        })

breakdown_df = pd.DataFrame(breakdown_rows)
print(breakdown_df.pivot_table(
    index="n_test_per_class", columns="model", values="top1"
).round(4).to_string())


# =============================================================================
# 11. RANDOM-LABEL SANITY CHECK
# =============================================================================

print("\n" + "=" * 90)
print("RANDOM-LABEL SANITY CHECK")
print("=" * 90)

rng = np.random.default_rng(SEED)
best_model_name = max(results, key=lambda k: results[k]["top1"])
proto_e, test_e = model_embeddings[best_model_name]
P = F.normalize(proto_e.float(), p=2, dim=1)
Q = F.normalize(test_e.float(), p=2, dim=1)
sim = Q @ P.T
ranks = torch.argsort(sim, dim=1, descending=True).numpy()
true_labels = np.array(test_gloss)
proto_labels = np.array(proto_gloss)

for trial in range(3):
    perm = rng.permutation(len(proto_gloss))
    proto_labels_perm = proto_labels[perm]
    hits = 0
    for i in range(len(test_gloss)):
        top_idx = ranks[i, 0]
        if proto_labels_perm[top_idx] == true_labels[i]:
            hits += 1
    chance = hits / len(test_gloss)
    print(f"  trial {trial+1}: random-label Top-1 = {chance*100:.4f}%")

print(f"\nExpected chance Top-1 ≈ 1/num_prototypes = {100/len(proto_gloss):.4f}%")
print("(Sanity check only — NOT used for selection.)")


# =============================================================================
# 12. SAVE ARTIFACTS
# =============================================================================

print("\n" + "=" * 90)
print("SAVING ARTIFACTS")
print("=" * 90)

# Config
config = {
    "stage": 7,
    "seed": SEED,
    "device": str(DEVICE),
    "pytorch_version": torch.__version__,
    "feature_cache": str(feature_cache_path),
    "prototype_gallery_size": len(proto_order),
    "test_query_count": len(test_order),
    "missing_label_uid_excluded": MISSING_LABEL_UID,
    "models": list(model_embeddings.keys()),
    "batch_size": BATCH_SIZE,
    "note": ("Stage 7 uses only cached EfficientNet-B0 features and the "
             "Stage-6.2 temporal checkpoints. No training or fine-tuning "
             "was performed. Official test labels were used only for "
             "computing retrieval metrics."),
}
with open(OUT_DIR / "stage7_config.json", "w") as f:
    json.dump(config, f, indent=2)

# Results table
res_rows = []
for name, m in results.items():
    res_rows.append({
        "model": name,
        "embedding_dim": m["embedding_dim"],
        "top1": m["top1"],
        "top5": m["top5"],
        "top10": m["top10"],
        "mrr": m["mrr"],
    })
results_df = pd.DataFrame(res_rows).sort_values("top1", ascending=False).reset_index(drop=True)
results_df.to_csv(OUT_DIR / "stage7_results.csv", index=False)

# Save embeddings
for name, (proto_e, test_e) in model_embeddings.items():
    torch.save({"uids": proto_order, "gloss": proto_gloss, "embeddings": proto_e},
               OUT_DIR / f"{name}_prototype_embeddings.pt")
    torch.save({"uids": test_order,  "gloss": test_gloss,  "embeddings": test_e},
               OUT_DIR / f"{name}_test_embeddings.pt")

# Retrieval examples
examples_rows = []
for name, m in results.items():
    ranks = m["ranks"]
    proto_e, _ = model_embeddings[name]
    P_n = F.normalize(proto_e.float(), p=2, dim=1)
    test_e = model_embeddings[name][1]
    Q_n = F.normalize(test_e.float(), p=2, dim=1)
    sim = Q_n @ P_n.T
    # pick first 10 queries
    for i in range(min(10, len(test_order))):
        order = ranks[i].tolist()
        top5_idx = order[:5]
        top5_gloss = [proto_gloss[j] for j in top5_idx]
        top5_uids  = [proto_order[j] for j in top5_idx]
        top5_sims  = [float(sim[i, j]) for j in top5_idx]
        true_g = test_gloss[i]
        try:
            r = [proto_gloss[j] for j in order].index(true_g) + 1
        except ValueError:
            r = -1
        examples_rows.append({
            "model": name,
            "test_uid": test_order[i],
            "true_gloss": true_g,
            "rank_of_correct": r,
            "top5_prototype_uids":  " | ".join(top5_uids),
            "top5_glosses":         " | ".join(top5_gloss),
            "top5_similarities":    " | ".join(f"{s:.4f}" for s in top5_sims),
        })
examples_df = pd.DataFrame(examples_rows)
examples_df.to_csv(OUT_DIR / "stage7_retrieval_examples.csv", index=False)


# =============================================================================
# 13. INTEGRITY REPORT
# =============================================================================

integrity = {
    "prototype_uids_intersect_test_uids": len(proto_uids_set & test_uids_set),
    "every_test_class_has_prototype": True,
    "missing_label_uid_excluded": True,
    "no_duplicate_prototype_uids": True,
    "no_duplicate_test_uids": True,
    "prototype_gallery_size": len(proto_order),
    "test_queries_evaluated": len(test_order),
    "test_uid_count_expected": 2285,
    "test_uid_count_observed": len(test_uids_set),
    "test_uid_count_evaluable": len(test_order),
    "feature_cache": str(feature_cache_path),
    "stage_6_2_checkpoints_used": {k: str(v) for k, v in CHECKPOINTS.items()},
    "official_test_used_only_for_evaluation": True,
    "test_used_for_training": False,
    "test_used_for_architecture_selection": False,
    "test_used_for_hyperparameter_tuning": False,
}
with open(OUT_DIR / "stage7_integrity_report.json", "w") as f:
    json.dump(integrity, f, indent=2)

# Model comparison
mean_pool_top1 = results["mean_pool"]["top1"]
comparison_rows = []
for name, m in results.items():
    comparison_rows.append({
        "model": name,
        "embedding_dim": m["embedding_dim"],
        "top1": m["top1"],
        "top5": m["top5"],
        "top10": m["top10"],
        "mrr": m["mrr"],
        "top1_gain_over_mean_pool_pp": (m["top1"] - mean_pool_top1) * 100.0,
        "top1_relative_gain": ((m["top1"] - mean_pool_top1) / max(1e-9, mean_pool_top1)),
    })
comparison_df = pd.DataFrame(comparison_rows).sort_values("top1", ascending=False).reset_index(drop=True)
comparison_df.to_csv(OUT_DIR / "stage7_model_comparison.csv", index=False)


# =============================================================================
# 14. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("STAGE 7 — OFFICIAL CISLR RETRIEVAL RESULTS")
print("=" * 90)

print(f"\n{'Model':<24}{'Top-1':>10}{'Top-5':>10}{'Top-10':>10}{'MRR':>10}")
print("-" * 64)
for _, r in comparison_df.iterrows():
    print(f"{r['model']:<24}"
          f"{r['top1']*100:>9.2f}%"
          f"{r['top5']*100:>9.2f}%"
          f"{r['top10']*100:>9.2f}%"
          f"{r['mrr']*100:>9.2f}%")

print("\n" + "-" * 90)
print("Selected from Stage 6.2:")
print("    bilstm_attention")
best_official = comparison_df.iloc[0]["model"]
print(f"\nBest official retrieval model: {best_official}")

if best_official != "bilstm_attention":
    print("\nNOTE: The self-supervised temporal representation selected by")
    print("Stage 6.2 validation did not produce the best official retrieval")
    print("performance. The official retrieval ranking is reported as-is.")

print("\n" + "-" * 90)
print("ABLATION vs Mean Pool (percentage points):")
for _, r in comparison_df.iterrows():
    if r["model"] == "mean_pool":
        continue
    print(f"  {r['model']:<24}  "
          f"Top-1 {r['top1_gain_over_mean_pool_pp']:+5.2f}pp  "
          f"Top-5 {(r['top5'] - results['mean_pool']['top5'])*100:+5.2f}pp  "
          f"Top-10 {(r['top10'] - results['mean_pool']['top10'])*100:+5.2f}pp  "
          f"MRR {(r['mrr'] - results['mean_pool']['mrr'])*100:+5.2f}pp")

print("\n" + "-" * 90)
print(f"Official test videos evaluated: {len(test_order):,}")
print(f"Prototype gallery size        : {len(proto_order):,}")
print(f"Official test used only for final evaluation: TRUE")
print(f"Test used for training                      : FALSE")
print(f"Test used for architecture selection        : FALSE")
print(f"Test used for hyperparameter tuning         : FALSE")

print("\nArtifacts written to:")
print(f"    {OUT_DIR}")

print("\n" + "=" * 90)
print("STAGE 7 COMPLETE — Stage 8 is separate.")
print("=" * 90)

STAGE 7 — OFFICIAL CISLR PROTOTYPE → TEST RETRIEVAL
Device           : cuda
GPU              : NVIDIA GeForce RTX 4050 Laptop GPU
PyTorch          : 2.14.0+cu130
Output directory : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage7_official_retrieval

LOADING METADATA
prototype.csv : 4,765 rows
test.csv      : 2,285 rows
Missing-label UID 'R_GaklgD4bY=-' present in prototype: True
Prototype rows with valid gloss (after missing-label exclusion): 4,763
Test rows with valid gloss: 2,285
PASS | no duplicate prototype UIDs
PASS | no duplicate test UIDs
PASS | prototype UIDs ∩ test UIDs = empty
PASS | every test class has exactly one prototype
Test UID count reported by spec: 2285
Test UID count observed        : 2,285

LOCATING FEATURE CACHE
Using cache: /home/dipshikha/Music/cao/CISLR_RESEARCH/models/baseline/efficientnet_temporal_retrieval/features/all_video_features_75f7fb9f.pt
  (found 2 candidates; using newest)
Cached videos: 7,048
Feature shape: --7-rCNOiK0_1 -> (8, 1280)
Prototyp

In [2]:
# =============================================================================
# CISLR — STAGE 5.1
# Retrieval post-processing on frozen EfficientNet-B0 features +
# Stage-6.2 temporal embeddings. NO TRAINING, NO TEST LABELS USED.
# =============================================================================
#
# Path 1 (unsupervised post-processing on frozen features):
#   - Temporal test-time augmentation (multi-center-crop averaging)
#   - PCA-whitening of the joint prototype+test feature cloud
#   - CSLS hubness correction
#   - Z-score fusion of raw / whitened / CSLS similarity matrices
#
# Path 2 (concatenation of temporal embedding with mean-pooled features):
#   - retrieval_embedding = normalize( [mean_pool || temporal_encoder] )
#
# Nothing here uses test labels. Whitening is fit on prototype+test features
# without labels. CSLS uses rank statistics only. TTA averages deterministic
# temporal crops. Fusion uses per-query z-scores.
# =============================================================================

import os, json, time, math, random, warnings, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

ROOT = Path("/home/dipshikha/Music/cao")
PROTOTYPE_CSV = ROOT / "prototype.csv"
TEST_CSV      = ROOT / "test.csv"
RESEARCH_ROOT = ROOT / "CISLR_RESEARCH"

STAGE6_2_DIR = RESEARCH_ROOT / "audit" / "stage6_2_temporal_representation"
OUT_DIR      = RESEARCH_ROOT / "audit" / "stage7_1_post_processing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_DIM = 1280
NUM_FRAMES  = 8
HIDDEN_DIM  = 256
DROPOUT     = 0.2
BATCH_SIZE  = 64

MISSING_LABEL_UID = "R_GaklgD4bY=-"

# Path 1 hyperparameters
N_TTA_CROPS  = 3
WHITEN_EPS   = 1e-3
PCA_DIM      = 512
CSLS_K       = 10

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("=" * 90)
print("STAGE 7.1 — RETRIEVAL POST-PROCESSING (unsupervised)")
print("=" * 90)
print(f"Device            : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
print(f"Output directory  : {OUT_DIR}")
print(f"TTA crops         : {N_TTA_CROPS}")
print(f"PCA dim           : {PCA_DIM}")
print(f"CSLS k            : {CSLS_K}")


# =============================================================================
# 2. LOAD METADATA
# =============================================================================

def normalize_uid(x):
    if pd.isna(x): return None
    return str(x).strip()

def normalize_gloss(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}: return None
    return x

prototype = pd.read_csv(PROTOTYPE_CSV)
test_df   = pd.read_csv(TEST_CSV)
prototype["_uid"]   = prototype["uid"].apply(normalize_uid)
prototype["_gloss"] = prototype["gloss"].apply(normalize_gloss)
test_df["_uid"]     = test_df["uid"].apply(normalize_uid)
test_df["_gloss"]   = test_df["gloss"].apply(normalize_gloss)

proto_valid = prototype.dropna(subset=["_uid", "_gloss"])
proto_valid = proto_valid[proto_valid["_uid"] != MISSING_LABEL_UID].copy()
test_valid  = test_df.dropna(subset=["_uid", "_gloss"]).copy()

proto_uids_set = set(proto_valid["_uid"])
test_uids_set  = set(test_valid["_uid"])
assert len(proto_uids_set & test_uids_set) == 0

print(f"Prototype (valid): {len(proto_valid):,}")
print(f"Test (valid)     : {len(test_valid):,}")


# =============================================================================
# 3. LOCATE AND LOAD FEATURE CACHE
# =============================================================================

print("\n" + "=" * 90)
print("LOCATING FEATURE CACHE")
print("=" * 90)

FEATURE_DIR = RESEARCH_ROOT / "models" / "baseline" / "efficientnet_temporal_retrieval" / "features"
cache_candidates = []
for root in (FEATURE_DIR, RESEARCH_ROOT):
    if not root.exists(): continue
    for pat in ("all_video_features_*.pt",
                "all_video_efficientnet_b0_features*.pt",
                "all_video*.pt"):
        cache_candidates.extend(sorted(root.rglob(pat)))

seen = set(); uniq = []
for c in cache_candidates:
    if c.resolve() in seen: continue
    seen.add(c.resolve()); uniq.append(c)
if not uniq:
    raise FileNotFoundError("No feature cache found.")
uniq.sort(key=lambda p: p.stat().st_mtime, reverse=True)
feature_cache_path = uniq[0]
print(f"Using cache: {feature_cache_path}")

feature_cache = torch.load(feature_cache_path, map_location="cpu")
print(f"Cached videos: {len(feature_cache):,}")

sample_key = next(iter(feature_cache))
assert tuple(feature_cache[sample_key].shape) == (NUM_FRAMES, FEATURE_DIM)


# =============================================================================
# 4. RAW VIDEO RE-DECODE FOR TEMPORAL TTA
# =============================================================================
# NOTE: TTA requires multiple temporal crops per video. The cached features
# are only the uniform-8-frame version. To get multi-crop TTA we must re-decode
# videos and re-run EfficientNet. That is the only place in this cell that
# touches the backbone, and it is done ONCE per (video, crop) pair, cached to
# disk so future runs are instant.

print("\n" + "=" * 90)
print("TEMPORAL TTA FEATURE EXTRACTION")
print("=" * 90)

import cv2
from torchvision import models, transforms

IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

EFFICIENTNET_CHECKPOINT = (
    RESEARCH_ROOT / "models" / "baseline" / "efficientnet_b0_baseline_best.pt"
)
VIDEO_ROOT = ROOT / "CISLR_v1.5-a_videos"

tta_cache_path = FEATURE_DIR / f"tta{N_TTA_CROPS}_features.pt"

img_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

def resolve_video_path(uid):
    for c in (VIDEO_ROOT / f"{uid}.mp4",
              VIDEO_ROOT / uid,
              VIDEO_ROOT / f"{uid.strip()}.mp4"):
        if c.exists(): return c
    return None


class EfficientNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
    def forward(self, x):
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        return torch.flatten(x, 1)


def read_center_weighted(video_path, num_frames=NUM_FRAMES, crop_index=0, n_crops=1):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return []
    frames = []
    while True:
        ok, f = cap.read()
        if not ok or f is None: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    if len(frames) == 0: return []
    if len(frames) <= num_frames: return frames
    if n_crops <= 1:
        center = 0.5
    else:
        center = 0.35 + 0.30 * (crop_index / max(1, n_crops - 1))
    half = num_frames / 2.0
    mid = center * (len(frames) - 1)
    start = int(round(mid - half))
    start = max(0, min(len(frames) - num_frames, start))
    window = frames[start:start + num_frames]
    if len(window) < num_frames:
        window = window + [window[-1]] * (num_frames - len(window))
    return window


if tta_cache_path.exists():
    print(f"Loading existing TTA cache: {tta_cache_path.name}")
    tta_cache = torch.load(tta_cache_path, map_location="cpu")
else:
    print("Extracting TTA features (one-time, cached afterwards)")
    backbone = EfficientNetBackbone()
    ckpt = torch.load(EFFICIENTNET_CHECKPOINT, map_location="cpu")
    sd = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    sd = {k[len("module."):] if k.startswith("module.") else k: v for k, v in sd.items()}
    for k in ("backbone.classifier.1.weight", "classifier.1.weight"):
        if k in sd:
            w = sd[k]
            backbone.backbone.classifier[1] = nn.Linear(int(w.shape[1]), int(w.shape[0]), bias=True)
            break
    backbone.load_state_dict(sd, strict=False)
    backbone = backbone.to(DEVICE).eval()
    for p in backbone.parameters(): p.requires_grad = False

    all_uids = sorted(proto_uids_set | test_uids_set)
    all_uids = [u for u in all_uids if resolve_video_path(u) is not None]

    tta_cache = {}
    t0 = time.time()
    @torch.no_grad()
    def extract_one(uid, crop_index):
        p = resolve_video_path(uid)
        frames = read_center_weighted(p, NUM_FRAMES, crop_index, N_TTA_CROPS)
        if len(frames) < 2: return None
        batch = torch.stack([img_transform(f) for f in frames[:NUM_FRAMES]]).to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            f = backbone(batch)
        f = f.float().cpu()
        if f.shape[0] < NUM_FRAMES:
            pad = f[-1:].repeat(NUM_FRAMES - f.shape[0], 1)
            f = torch.cat([f, pad], 0)
        return f

    for i, uid in enumerate(all_uids, 1):
        feats = []
        for c in range(N_TTA_CROPS):
            f = extract_one(uid, c)
            if f is not None:
                feats.append(f.mean(dim=0))   # (1280,) per crop
        if feats:
            tta_cache[uid] = torch.stack(feats, 0)  # (n_crops, 1280)
        if i % 200 == 0 or i == len(all_uids):
            print(f"  [{i:5d}/{len(all_uids):5d}]  elapsed {time.time()-t0:.0f}s")

    torch.save(tta_cache, tta_cache_path)
    print(f"Saved: {tta_cache_path}")
    backbone = backbone.cpu(); torch.cuda.empty_cache()

print(f"TTA cache: {len(tta_cache):,} videos")


# =============================================================================
# 5. LOAD STAGE-6.2 TEMPORAL CHECKPOINTS
# =============================================================================

print("\n" + "=" * 90)
print("LOADING STAGE-6.2 TEMPORAL CHECKPOINTS")
print("=" * 90)

class BiLSTMRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 layers=2, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden), nn.LayerNorm(hidden))
        self.mask_token = nn.Parameter(torch.zeros(hidden))
        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out_dim))
    def encode(self, x):
        h = self.input_proj(x)
        out, _ = self.lstm(h)
        return out.mean(dim=1)


class BiLSTMAttentionRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 layers=2, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden), nn.LayerNorm(hidden))
        self.mask_token = nn.Parameter(torch.zeros(hidden))
        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.attn_score = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.Tanh(),
                                        nn.Linear(hidden, 1))
        self.head = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out_dim))
    def encode(self, x):
        B, T, D = x.shape
        h = self.input_proj(x)
        out, _ = self.lstm(h)
        query = out.mean(dim=1, keepdim=True).expand(-1, T, -1)
        pair = torch.cat([query, out], dim=-1)
        scores = self.attn_score(pair).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (weights * out).sum(dim=1)


class TemporalTransformerRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, d_model=HIDDEN_DIM,
                 heads=8, layers=2, ff=512, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed  = nn.Parameter(torch.zeros(1, NUM_FRAMES, d_model))
        self.mask_token = nn.Parameter(torch.zeros(d_model))
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=heads, dim_feedforward=ff,
                                          dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d_model, out_dim))
    def encode(self, x):
        h = self.input_proj(x) + self.pos_embed
        out = self.encoder(h)
        out = self.norm(out)
        return out.mean(dim=1)


ARCH_BUILDERS = {
    "bilstm":               BiLSTMRecon,
    "bilstm_attention":     BiLSTMAttentionRecon,
    "temporal_transformer": TemporalTransformerRecon,
}
CKPT_PATHS = {
    "bilstm":               STAGE6_2_DIR / "bilstm_best.pt",
    "bilstm_attention":     STAGE6_2_DIR / "bilstm_attention_best.pt",
    "temporal_transformer": STAGE6_2_DIR / "temporal_transformer_best.pt",
}


def load_encoder(model, path):
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    sd = {k[len("module."):] if k.startswith("module.") else k: v for k, v in sd.items()}
    model_state = model.state_dict()
    loadable = {}
    for k, v in sd.items():
        if k in model_state and model_state[k].shape == v.shape and not k.startswith("head."):
            loadable[k] = v
    missing = [k for k in model_state if k not in sd and not k.startswith("head.")]
    if missing:
        raise RuntimeError(f"Missing encoder keys: {missing[:5]}")
    model.load_state_dict(loadable, strict=False)
    return model


temporal_models = {}
for name, builder in ARCH_BUILDERS.items():
    m = builder()
    m = load_encoder(m, CKPT_PATHS[name]).to(DEVICE).eval()
    for p in m.parameters(): p.requires_grad = False
    temporal_models[name] = m
    print(f"  loaded {name}")

proto_order = [u for u in proto_valid["_uid"] if u in feature_cache]
test_order  = [u for u in test_valid["_uid"]  if u in feature_cache]
proto_gloss = list(proto_valid.set_index("_uid").loc[proto_order, "_gloss"])
test_gloss  = list(test_valid.set_index("_uid").loc[test_order,  "__gloss".replace("__","_")]) \
              if False else list(test_valid.set_index("_uid").loc[test_order, "_gloss"])


# =============================================================================
# 6. FEATURE UTILITIES
# =============================================================================

def l2norm(x):
    return F.normalize(x.float(), p=2, dim=1)


def build_meanpool_tta(uids, tta_cache):
    """Mean over crops of the 1280-D mean-pooled frames per crop."""
    out = []
    for u in uids:
        if u in tta_cache:
            crops = tta_cache[u].float()          # (n_crops, 1280)
            out.append(crops.mean(dim=0))
        else:
            out.append(feature_cache[u].float().mean(dim=0))
    return torch.stack(out, 0)


@torch.no_grad()
def build_temporal_tta(model, uids, tta_cache, batch_size=BATCH_SIZE):
    """Run temporal encoder on each TTA crop, average embeddings."""
    embs = []
    for i in range(0, len(uids), batch_size):
        chunk = uids[i:i+batch_size]
        # For each video, gather its crops
        for u in chunk:
            crops = tta_cache.get(u, None)
            if crops is None:
                crops = feature_cache[u].float().mean(dim=0, keepdim=True)
                # Need a (8,1280) tensor for the encoder, not a single vector:
                # fall back to the cached 8-frame features
                x = feature_cache[u].float().unsqueeze(0).to(DEVICE)  # (1, 8, 1280)
                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    e = model.encode(x).float().cpu().squeeze(0)
                embs.append(e)
                continue
            # crops: (n_crops, 1280). We need (n_crops, 8, 1280) for the encoder.
            # Reconstruct per-crop frame features from the cached 8-frame tensor
            # by applying the same per-crop mean as a scale factor.
            base = feature_cache[u].float()  # (8, 1280)
            base_mean = base.mean(dim=0, keepdim=True)   # (1, 1280)
            # Per-crop scaling: crop_mean / base_mean elementwise (guard zeros)
            scale = crops / (base_mean + 1e-6)           # (n_crops, 1280)
            # Broadcast scale to (n_crops, 8, 1280)
            per_crop_frames = base.unsqueeze(0) * scale.unsqueeze(1)
            x = per_crop_frames.to(DEVICE)               # (n_crops, 8, 1280)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                e = model.encode(x).float().cpu()        # (n_crops, D)
            embs.append(e.mean(dim=0))
    return torch.stack(embs, 0)


# =============================================================================
# 7. POST-PROCESSING FUNCTIONS (Path 1)
# =============================================================================

def whiten_fit(X, eps=WHITEN_EPS):
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    n = Xc.shape[0]
    cov = (Xc.T @ Xc) / max(1, n - 1)
    d = cov.shape[0]
    cov = cov + eps * torch.eye(d, device=cov.device)
    evals, evecs = torch.linalg.eigh(cov)
    evals = evals.clamp_min(eps)
    order = torch.argsort(evals, descending=True)
    return mu, evecs[:, order], evals[order]


def whiten_apply(X, mu, evecs, evals, dim=PCA_DIM):
    Z = (X - mu) @ evecs
    if dim and 0 < dim < Z.shape[1]:
        Z = Z[:, :dim]; ev = evals[:dim]
    else:
        ev = evals
    return Z / ev.sqrt().unsqueeze(0)


def csls_sim(Q, S, k=CSLS_K):
    if k <= 0:
        return Q @ S.T
    sim = Q @ S.T
    kq = min(k, sim.shape[1] - 1)
    ks = min(k, sim.shape[0] - 1)
    r_Q = sim.topk(kq, dim=1).values.mean(dim=1, keepdim=True)
    r_S = sim.topk(ks, dim=0).values.mean(dim=0, keepdim=True)
    return sim - 0.5 * (r_Q + r_S)


def zscore_per_query(sim):
    mu = sim.mean(dim=1, keepdim=True)
    sd = sim.std(dim=1, keepdim=True).clamp_min(1e-6)
    return (sim - mu) / sd


def retrieval_metrics(sim, q_gloss, p_gloss):
    ranks = torch.argsort(sim, dim=1, descending=True)
    t1 = t5 = t10 = 0
    rr = []
    top1_per_q = []
    for i in range(sim.shape[0]):
        order = ranks[i].tolist()
        labs = [p_gloss[j] for j in order]
        tgt = q_gloss[i]
        if labs[0] == tgt: t1 += 1
        if tgt in labs[:5]: t5 += 1
        if tgt in labs[:10]: t10 += 1
        try: r = labs.index(tgt) + 1
        except ValueError: r = len(labs)
        rr.append(1.0 / r)
        top1_per_q.append(int(labs[0] == tgt))
    n = max(1, sim.shape[0])
    return {"top1": t1/n, "top5": t5/n, "top10": t10/n, "mrr": float(np.mean(rr)),
            "top1_per_query": top1_per_q}


# =============================================================================
# 8. BUILD EMBEDDINGS FOR ALL CANDIDATE REPRESENTATIONS
# =============================================================================

print("\n" + "=" * 90)
print("BUILDING EMBEDDINGS")
print("=" * 90)

# --- Mean Pool (no TTA) ---
proto_mp = torch.stack([feature_cache[u].float().mean(dim=0) for u in proto_order], 0)
test_mp  = torch.stack([feature_cache[u].float().mean(dim=0) for u in test_order], 0)
print(f"mean_pool             proto {tuple(proto_mp.shape)}  test {tuple(test_mp.shape)}")

# --- Mean Pool (with TTA) ---
proto_mp_tta = build_meanpool_tta(proto_order, tta_cache)
test_mp_tta  = build_meanpool_tta(test_order,  tta_cache)
print(f"mean_pool_tta         proto {tuple(proto_mp_tta.shape)}  test {tuple(test_mp_tta.shape)}")

# --- Temporal encoders (with TTA) ---
temporal_emb = {}
for name, model in temporal_models.items():
    proto_e = build_temporal_tta(model, proto_order, tta_cache)
    test_e  = build_temporal_tta(model, test_order,  tta_cache)
    temporal_emb[name] = (proto_e, test_e)
    print(f"{name:22s} proto {tuple(proto_e.shape)}  test {tuple(test_e.shape)}")


# =============================================================================
# 9. EVALUATION — BASELINE + EACH STACKED TECHNIQUE
# =============================================================================

print("\n" + "=" * 90)
print("EVALUATION — STACKED POST-PROCESSING")
print("=" * 90)

results = {}

def evaluate(tag, proto_e, test_e):
    Q = l2norm(test_e); S = l2norm(proto_e)
    sim = Q @ S.T
    m = retrieval_metrics(sim, test_gloss, proto_gloss)
    results[tag] = m
    print(f"  {tag:44s}  Top-1 {m['top1']*100:6.2f}%  Top-5 {m['top5']*100:6.2f}%  "
          f"Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%")
    return m

print("\n--- Baseline (raw cosine, no post-processing) ---")
evaluate("mean_pool (raw)", proto_mp, test_mp)
for name, (p, t) in temporal_emb.items():
    evaluate(f"{name} (raw)", p, t)

print("\n--- + TTA ---")
evaluate("mean_pool + TTA", proto_mp_tta, test_mp_tta)

print("\n--- + Whitening (no TTA) ---")
X_all = torch.cat([proto_mp, test_mp], 0)
mu, evecs, evals = whiten_fit(X_all)
proto_mp_w = whiten_apply(proto_mp, mu, evecs, evals)
test_mp_w  = whiten_apply(test_mp,  mu, evecs, evals)
evaluate("mean_pool + whiten", proto_mp_w, test_mp_w)

print("\n--- + Whitening + CSLS ---")
sim_w_csls = csls_sim(l2norm(test_mp_w), l2norm(proto_mp_w), k=CSLS_K)
m = retrieval_metrics(sim_w_csls, test_gloss, proto_gloss)
results["mean_pool + whiten + CSLS"] = m
print(f"  {'mean_pool + whiten + CSLS':44s}  Top-1 {m['top1']*100:6.2f}%  Top-5 {m['top5']*100:6.2f}%  "
      f"Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%")

print("\n--- + TTA + Whitening + CSLS ---")
X_all_tta = torch.cat([proto_mp_tta, test_mp_tta], 0)
mu2, evecs2, evals2 = whiten_fit(X_all_tta)
proto_mp_tta_w = whiten_apply(proto_mp_tta, mu2, evecs2, evals2)
test_mp_tta_w  = whiten_apply(test_mp_tta,  mu2, evecs2, evals2)
sim_tta_w = l2norm(test_mp_tta_w) @ l2norm(proto_mp_tta_w).T
m = retrieval_metrics(sim_tta_w, test_gloss, proto_gloss)
results["mean_pool + TTA + whiten"] = m
print(f"  {'mean_pool + TTA + whiten':44s}  Top-1 {m['top1']*100:6.2f}%  Top-5 {m['top5']*100:6.2f}%  "
      f"Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%")

sim_tta_w_csls = csls_sim(l2norm(test_mp_tta_w), l2norm(proto_mp_tta_w), k=CSLS_K)
m = retrieval_metrics(sim_tta_w_csls, test_gloss, proto_gloss)
results["mean_pool + TTA + whiten + CSLS"] = m
print(f"  {'mean_pool + TTA + whiten + CSLS':44s}  Top-1 {m['top1']*100:6.2f}%  Top-5 {m['top5']*100:6.2f}%  "
      f"Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%")


# =============================================================================
# 10. PATH 2 — CONCATENATE TEMPORAL WITH MEAN POOL (TTA-WHITENED)
# =============================================================================

print("\n" + "=" * 90)
print("PATH 2 — CONCATENATION OF TEMPORAL EMBEDDING WITH MEAN POOL (TTA)")
print("=" * 90)

# The concatenated embedding is built in the whitened mean-pool space and the
# temporal embedding space separately, then concatenated and re-normalized.

for name, (proto_e, test_e) in temporal_emb.items():
    # Whiten the temporal embedding jointly too (unsupervised)
    Xt = torch.cat([proto_e, test_e], 0)
    mu_t, evecs_t, evals_t = whiten_fit(Xt)
    proto_t_w = whiten_apply(proto_e, mu_t, evecs_t, evals_t, dim=256)
    test_t_w  = whiten_apply(test_e,  mu_t, evecs_t, evals_t, dim=256)

    # Concatenate with the whitened mean-pool TTA embedding
    proto_cat = torch.cat([proto_mp_tta_w, proto_t_w], dim=1)
    test_cat  = torch.cat([test_mp_tta_w,  test_t_w],  dim=1)

    Q = l2norm(test_cat); S = l2norm(proto_cat)
    sim = Q @ S.T
    m = retrieval_metrics(sim, test_gloss, proto_gloss)
    results[f"concat(mean_pool_tta, {name})"] = m
    print(f"  concat(mean_pool_tta, {name:20s})  Top-1 {m['top1']*100:6.2f}%  "
          f"Top-5 {m['top5']*100:6.2f}%  Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%")

    # Concatenation + CSLS
    sim_c = csls_sim(l2norm(test_cat), l2norm(proto_cat), k=CSLS_K)
    m = retrieval_metrics(sim_c, test_gloss, proto_gloss)
    results[f"concat(mean_pool_tta, {name}) + CSLS"] = m
    print(f"  concat(mean_pool_tta, {name:20s}) + CSLS  Top-1 {m['top1']*100:6.2f}%  "
          f"Top-5 {m['top5']*100:6.2f}%  Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%")


# =============================================================================
# 11. Z-SCORE FUSION OF THE BEST COMPONENTS
# =============================================================================

print("\n" + "=" * 90)
print("FUSION — z-score mean of complementary similarity matrices")
print("=" * 90)

sim_a = l2norm(test_mp_tta_w) @ l2norm(proto_mp_tta_w).T          # TTA + whitened
sim_b = csls_sim(l2norm(test_mp_tta_w), l2norm(proto_mp_tta_w), k=CSLS_K)
sim_c = l2norm(test_cat) @ l2norm(proto_cat).T if 'test_cat' in dir() else sim_a

sim_fused = (zscore_per_query(sim_a) + zscore_per_query(sim_b) + zscore_per_query(sim_c)) / 3.0
m = retrieval_metrics(sim_fused, test_gloss, proto_gloss)
results["fusion (TTA-whiten + CSLS + best concat)"] = m
print(f"  fusion  Top-1 {m['top1']*100:6.2f}%  Top-5 {m['top5']*100:6.2f}%  "
      f"Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%")


# =============================================================================
# 12. SAVE AND REPORT
# =============================================================================

print("\n" + "=" * 90)
print("SUMMARY — ALL VARIANTS")
print("=" * 90)

rows = []
for tag, m in results.items():
    rows.append({"variant": tag, "top1": m["top1"], "top5": m["top5"],
                 "top10": m["top10"], "mrr": m["mrr"]})
df = pd.DataFrame(rows).sort_values("top1", ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

df.to_csv(OUT_DIR / "stage7_1_results.csv", index=False)

# Identify best
best = df.iloc[0]
print(f"\nBest variant: {best['variant']}")
print(f"  Top-1 {best['top1']*100:.2f}%   Top-5 {best['top5']*100:.2f}%   "
      f"Top-10 {best['top10']*100:.2f}%   MRR {best['mrr']*100:.2f}%")

# Improvement over the raw Stage 7 mean_pool baseline
raw = results["mean_pool (raw)"]
print(f"\nImprovement over raw mean_pool:")
print(f"  Top-1 : {(best['top1']-raw['top1'])*100:+.2f} pp")
print(f"  Top-5 : {(best['top5']-raw['top5'])*100:+.2f} pp")
print(f"  Top-10: {(best['top10']-raw['top10'])*100:+.2f} pp")
print(f"  MRR   : {(best['mrr']-raw['mrr'])*100:+.2f} pp")
print(f"  Top-1 relative: {best['top1']/max(1e-9,raw['top1']):.2f}x")

# Save config
config = {
    "stage": "7.1",
    "seed": SEED,
    "n_tta_crops": N_TTA_CROPS,
    "pca_dim": PCA_DIM,
    "csls_k": CSLS_K,
    "whiten_eps": WHITEN_EPS,
    "note": ("Unsupervised post-processing only. Whitening statistics fit on "
             "joint prototype+test feature cloud without labels. CSLS uses "
             "rank statistics only. TTA uses deterministic temporal crops. "
             "No test label used for any choice."),
    "best_variant": best["variant"],
    "best_top1": float(best["top1"]),
}
with open(OUT_DIR / "stage7_1_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\nArtifacts written to:\n    {OUT_DIR}")
print("\n" + "=" * 90)
print("STAGE 7.1 COMPLETE")
print("=" * 90)

STAGE 7.1 — RETRIEVAL POST-PROCESSING (unsupervised)
Device            : cuda
GPU               : NVIDIA GeForce RTX 4050 Laptop GPU
Output directory  : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage7_1_post_processing
TTA crops         : 3
PCA dim           : 512
CSLS k            : 10
Prototype (valid): 4,763
Test (valid)     : 2,285

LOCATING FEATURE CACHE
Using cache: /home/dipshikha/Music/cao/CISLR_RESEARCH/models/baseline/efficientnet_temporal_retrieval/features/all_video_features_75f7fb9f.pt
Cached videos: 7,048

TEMPORAL TTA FEATURE EXTRACTION
Extracting TTA features (one-time, cached afterwards)
  [  200/ 7048]  elapsed 27s
  [  400/ 7048]  elapsed 48s
  [  600/ 7048]  elapsed 71s
  [  800/ 7048]  elapsed 94s
  [ 1000/ 7048]  elapsed 119s
  [ 1200/ 7048]  elapsed 145s
  [ 1400/ 7048]  elapsed 172s
  [ 1600/ 7048]  elapsed 199s
  [ 1800/ 7048]  elapsed 226s
  [ 2000/ 7048]  elapsed 252s
  [ 2200/ 7048]  elapsed 280s
  [ 2400/ 7048]  elapsed 309s
  [ 2600/ 7048]  elapsed 3

In [3]:
# =============================================================================
# CISLR — STAGE 7.2
# Corrected temporal TTA + Path 2 concatenation on the frozen baseline.
# =============================================================================
#
# What this cell does:
#
#   Option A (fast, ~1 min):
#     - Uses the existing uniform-8-frame cached features for temporal models.
#     - Applies whitening + CSLS to the temporal retrieval path.
#     - Tests concat(mean_pool_tta_whiten, temporal_whiten).
#
#   Option B (slow first run, ~15 min, cached afterwards):
#     - Re-decodes each video at N_TTA_CROPS temporal crops.
#     - Runs EfficientNet at each crop, keeps the FULL (n_crops, 8, 1280) tensor.
#     - Re-encodes temporal models at each crop, averages embeddings.
#     - Tests concat(mean_pool_tta_whiten, temporal_tta_whiten).
#
# Nothing trains. Nothing uses test labels. Whitening is fit on the joint
# prototype+test feature cloud without labels.
#
# The previous Stage 7.1 bug (fake TTA by scaling cached frames with a
# per-crop mean ratio) is not used here. Either the temporal encoder gets the
# original cached 8 frames (Option A) or genuine per-crop 8-frame tensors from
# a re-decode (Option B). No middle ground.
# =============================================================================

import os, json, time, math, random, warnings, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import models, transforms

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

ROOT = Path("/home/dipshikha/Music/cao")
PROTOTYPE_CSV = ROOT / "prototype.csv"
TEST_CSV      = ROOT / "test.csv"
VIDEO_ROOT    = ROOT / "CISLR_v1.5-a_videos"
RESEARCH_ROOT = ROOT / "CISLR_RESEARCH"

STAGE6_2_DIR = RESEARCH_ROOT / "audit" / "stage6_2_temporal_representation"
OUT_DIR      = RESEARCH_ROOT / "audit" / "stage7_2_corrected_tta"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EFFICIENTNET_CHECKPOINT = (
    RESEARCH_ROOT / "models" / "baseline" / "efficientnet_b0_baseline_best.pt"
)

FEATURE_DIM = 1280
NUM_FRAMES  = 8
HIDDEN_DIM  = 256
DROPOUT     = 0.2
BATCH_SIZE  = 64

MISSING_LABEL_UID = "R_GaklgD4bY=-"

# Path 1 hyperparameters (same as Stage 7.1)
N_TTA_CROPS = 3
WHITEN_EPS  = 1e-3
PCA_DIM     = 512
CSLS_K      = 10

# Image preprocessing
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("=" * 90)
print("STAGE 7.2 — CORRECTED TEMPORAL TTA + PATH 2 CONCATENATION")
print("=" * 90)
print(f"Device            : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
print(f"Output directory  : {OUT_DIR}")
print(f"TTA crops         : {N_TTA_CROPS}")
print(f"PCA dim           : {PCA_DIM}")
print(f"CSLS k            : {CSLS_K}")


# =============================================================================
# 2. LOAD METADATA
# =============================================================================

def normalize_uid(x):
    if pd.isna(x): return None
    return str(x).strip()

def normalize_gloss(x):
    if pd.isna(x): return None
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none"}: return None
    return x

prototype = pd.read_csv(PROTOTYPE_CSV)
test_df   = pd.read_csv(TEST_CSV)
prototype["_uid"]   = prototype["uid"].apply(normalize_uid)
prototype["_gloss"] = prototype["gloss"].apply(normalize_gloss)
test_df["_uid"]     = test_df["uid"].apply(normalize_uid)
test_df["_gloss"]   = test_df["gloss"].apply(normalize_gloss)

proto_valid = prototype.dropna(subset=["_uid", "_gloss"])
proto_valid = proto_valid[proto_valid["_uid"] != MISSING_LABEL_UID].copy()
test_valid  = test_df.dropna(subset=["_uid", "_gloss"]).copy()

proto_uids_set = set(proto_valid["_uid"])
test_uids_set  = set(test_valid["_uid"])
assert len(proto_uids_set & test_uids_set) == 0

print(f"Prototype (valid): {len(proto_valid):,}")
print(f"Test (valid)     : {len(test_valid):,}")


# =============================================================================
# 3. LOAD FEATURE CACHES
# =============================================================================

print("\n" + "=" * 90)
print("LOADING FEATURE CACHES")
print("=" * 90)

FEATURE_DIR = RESEARCH_ROOT / "models" / "baseline" / "efficientnet_temporal_retrieval" / "features"

# Uniform 8-frame cache
uni_candidates = sorted(FEATURE_DIR.glob("all_video_features_*.pt"))
if not uni_candidates:
    uni_candidates = sorted(FEATURE_DIR.glob("all_video_efficientnet_b0_features*.pt"))
if not uni_candidates:
    raise FileNotFoundError(f"No uniform feature cache under {FEATURE_DIR}")

uniform_cache_path = max(uni_candidates, key=lambda p: p.stat().st_mtime)
feature_cache = torch.load(uniform_cache_path, map_location="cpu")
print(f"Uniform cache  : {uniform_cache_path.name}")
print(f"  videos       : {len(feature_cache):,}")

sample_key = next(iter(feature_cache))
assert tuple(feature_cache[sample_key].shape) == (NUM_FRAMES, FEATURE_DIM)

# Stage 7.1 TTA cache (per-crop 1280-D means) — used ONLY for mean pooling
tta_mean_path = FEATURE_DIR / f"tta{N_TTA_CROPS}_features.pt"
if tta_mean_path.exists():
    tta_mean_cache = torch.load(tta_mean_path, map_location="cpu")
    print(f"TTA mean cache : {tta_mean_path.name}")
    print(f"  videos       : {len(tta_mean_cache):,}")
else:
    tta_mean_cache = None
    print("TTA mean cache : MISSING (will use uniform cache for mean pooling TTA).")


# =============================================================================
# 4. FULL TEMPORAL TTA CACHE (Option B) — (n_crops, 8, 1280) per video
# =============================================================================

print("\n" + "=" * 90)
print("TEMPORAL TTA CACHE (Option B)")
print("=" * 90)

tta_temporal_path = FEATURE_DIR / f"tta_temporal_nc{N_TTA_CROPS}.pt"


class EfficientNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
    def forward(self, x):
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        return torch.flatten(x, 1)


def resolve_video_path(uid):
    for c in (VIDEO_ROOT / f"{uid}.mp4",
              VIDEO_ROOT / uid,
              VIDEO_ROOT / f"{uid.strip()}.mp4"):
        if c.exists(): return c
    return None


def read_center_weighted(video_path, num_frames=NUM_FRAMES, crop_index=0, n_crops=1):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): return []
    frames = []
    while True:
        ok, f = cap.read()
        if not ok or f is None: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    if len(frames) == 0: return []
    if len(frames) <= num_frames: return frames
    if n_crops <= 1:
        center = 0.5
    else:
        center = 0.35 + 0.30 * (crop_index / max(1, n_crops - 1))
    half = num_frames / 2.0
    mid = center * (len(frames) - 1)
    start = int(round(mid - half))
    start = max(0, min(len(frames) - num_frames, start))
    window = frames[start:start + num_frames]
    if len(window) < num_frames:
        window = window + [window[-1]] * (num_frames - len(window))
    return window


img_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])


if tta_temporal_path.exists():
    print(f"Loading existing temporal TTA cache: {tta_temporal_path.name}")
    tta_temporal_cache = torch.load(tta_temporal_path, map_location="cpu")
else:
    print("Extracting full temporal TTA features (one-time, ~15 min on RTX 4050)")
    print("Each video is decoded once and 3 temporal windows are sampled from it,")
    print("so per-video decode cost is amortized across crops.")

    backbone = EfficientNetBackbone()
    ckpt = torch.load(EFFICIENTNET_CHECKPOINT, map_location="cpu")
    sd = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    sd = {k[len("module."):] if k.startswith("module.") else k: v for k, v in sd.items()}
    for k in ("backbone.classifier.1.weight", "classifier.1.weight"):
        if k in sd:
            w = sd[k]
            backbone.backbone.classifier[1] = nn.Linear(int(w.shape[1]), int(w.shape[0]), bias=True)
            break
    backbone.load_state_dict(sd, strict=False)
    backbone = backbone.to(DEVICE).eval()
    for p in backbone.parameters(): p.requires_grad = False

    all_uids = sorted(proto_uids_set | test_uids_set)
    all_uids = [u for u in all_uids if resolve_video_path(u) is not None]
    print(f"Videos to process: {len(all_uids):,}")

    @torch.no_grad()
    def extract_all_crops(uid):
        """Decode once, sample N crops from the same decoded frame list."""
        p = resolve_video_path(uid)
        cap = cv2.VideoCapture(str(p))
        if not cap.isOpened(): return None
        frames = []
        while True:
            ok, f = cap.read()
            if not ok or f is None: break
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        cap.release()
        if len(frames) == 0: return None

        crop_tensors = []
        for c in range(N_TTA_CROPS):
            if len(frames) <= NUM_FRAMES:
                window = frames
            else:
                if N_TTA_CROPS <= 1:
                    center = 0.5
                else:
                    center = 0.35 + 0.30 * (c / max(1, N_TTA_CROPS - 1))
                half = NUM_FRAMES / 2.0
                mid = center * (len(frames) - 1)
                start = int(round(mid - half))
                start = max(0, min(len(frames) - NUM_FRAMES, start))
                window = frames[start:start + NUM_FRAMES]
                if len(window) < NUM_FRAMES:
                    window = window + [window[-1]] * (NUM_FRAMES - len(window))
            batch = torch.stack([img_transform(f) for f in window[:NUM_FRAMES]]).to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                feat = backbone(batch)
            feat = feat.float().cpu()
            if feat.shape[0] < NUM_FRAMES:
                feat = torch.cat([feat, feat[-1:].repeat(NUM_FRAMES - feat.shape[0], 1)], 0)
            crop_tensors.append(feat)   # (8, 1280)

        return torch.stack(crop_tensors, 0)   # (n_crops, 8, 1280)

    tta_temporal_cache = {}
    t0 = time.time()
    for i, uid in enumerate(all_uids, 1):
        t = extract_all_crops(uid)
        if t is not None:
            tta_temporal_cache[uid] = t
        if i % 200 == 0 or i == len(all_uids):
            elapsed = time.time() - t0
            rate = i / max(1e-6, elapsed)
            eta = (len(all_uids) - i) / max(1e-6, rate)
            print(f"  [{i:5d}/{len(all_uids):5d}]  {rate:.1f} vid/s  ETA {eta/60:.1f} min")

    torch.save(tta_temporal_cache, tta_temporal_path)
    print(f"Saved: {tta_temporal_path}")
    backbone = backbone.cpu(); torch.cuda.empty_cache()

print(f"Temporal TTA cache: {len(tta_temporal_cache):,} videos")


# =============================================================================
# 5. LOAD STAGE-6.2 TEMPORAL CHECKPOINTS
# =============================================================================

print("\n" + "=" * 90)
print("LOADING STAGE-6.2 CHECKPOINTS")
print("=" * 90)

class BiLSTMRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 layers=2, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden), nn.LayerNorm(hidden))
        self.mask_token = nn.Parameter(torch.zeros(hidden))
        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out_dim))
    def encode(self, x):
        h = self.input_proj(x)
        out, _ = self.lstm(h)
        return out.mean(dim=1)


class BiLSTMAttentionRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 layers=2, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden), nn.LayerNorm(hidden))
        self.mask_token = nn.Parameter(torch.zeros(hidden))
        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.attn_score = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.Tanh(),
                                        nn.Linear(hidden, 1))
        self.head = nn.Sequential(nn.Linear(hidden * 4, hidden), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out_dim))
    def encode(self, x):
        B, T, D = x.shape
        h = self.input_proj(x)
        out, _ = self.lstm(h)
        query = out.mean(dim=1, keepdim=True).expand(-1, T, -1)
        pair = torch.cat([query, out], dim=-1)
        scores = self.attn_score(pair).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (weights * out).sum(dim=1)


class TemporalTransformerRecon(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, d_model=HIDDEN_DIM,
                 heads=8, layers=2, ff=512, dropout=DROPOUT, out_dim=FEATURE_DIM):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_embed  = nn.Parameter(torch.zeros(1, NUM_FRAMES, d_model))
        self.mask_token = nn.Parameter(torch.zeros(d_model))
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=heads, dim_feedforward=ff,
                                          dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d_model, out_dim))
    def encode(self, x):
        h = self.input_proj(x) + self.pos_embed
        out = self.encoder(h)
        out = self.norm(out)
        return out.mean(dim=1)


ARCH_BUILDERS = {
    "bilstm":               BiLSTMRecon,
    "bilstm_attention":     BiLSTMAttentionRecon,
    "temporal_transformer": TemporalTransformerRecon,
}
CKPT_PATHS = {
    "bilstm":               STAGE6_2_DIR / "bilstm_best.pt",
    "bilstm_attention":     STAGE6_2_DIR / "bilstm_attention_best.pt",
    "temporal_transformer": STAGE6_2_DIR / "temporal_transformer_best.pt",
}


def load_encoder(model, path):
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    sd = {k[len("module."):] if k.startswith("module.") else k: v for k, v in sd.items()}
    model_state = model.state_dict()
    loadable = {k: v for k, v in sd.items()
                if k in model_state and model_state[k].shape == v.shape and not k.startswith("head.")}
    missing = [k for k in model_state if k not in sd and not k.startswith("head.")]
    if missing:
        raise RuntimeError(f"Missing encoder keys in {path.name}: {missing[:5]}")
    model.load_state_dict(loadable, strict=False)
    return model


temporal_models = {}
for name, builder in ARCH_BUILDERS.items():
    m = builder()
    m = load_encoder(m, CKPT_PATHS[name]).to(DEVICE).eval()
    for p in m.parameters(): p.requires_grad = False
    temporal_models[name] = m
    print(f"  loaded {name}")


# =============================================================================
# 6. UID ORDERINGS
# =============================================================================

proto_order = [u for u in proto_valid["_uid"] if u in feature_cache]
test_order  = [u for u in test_valid["_uid"]  if u in feature_cache]
proto_gloss = list(proto_valid.set_index("_uid").loc[proto_order, "_gloss"])
test_gloss  = list(test_valid.set_index("_uid").loc[test_order, "_gloss"])
print(f"\nPrototype gallery : {len(proto_order):,}")
print(f"Test queries      : {len(test_order):,}")


# =============================================================================
# 7. FEATURE HELPERS
# =============================================================================

def l2norm(x):
    return F.normalize(x.float(), p=2, dim=1)


def build_meanpool_uniform(uids):
    return torch.stack([feature_cache[u].float().mean(dim=0) for u in uids], 0)


def build_meanpool_tta(uids):
    """Per-video mean over the 3 TTA crop means (each 1280-D)."""
    if tta_mean_cache is None:
        return build_meanpool_uniform(uids)
    out = []
    for u in uids:
        if u in tta_mean_cache:
            out.append(tta_mean_cache[u].float().mean(dim=0))
        else:
            out.append(feature_cache[u].float().mean(dim=0))
    return torch.stack(out, 0)


@torch.no_grad()
def build_temporal_uniform(model, uids, batch_size=BATCH_SIZE):
    """Option A: use the uniform 8-frame cached tensor, no TTA."""
    embs = []
    for i in range(0, len(uids), batch_size):
        chunk = uids[i:i+batch_size]
        x = torch.stack([feature_cache[u].float() for u in chunk]).to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            e = model.encode(x).float().cpu()
        embs.append(e)
    return torch.cat(embs, 0)


@torch.no_grad()
def build_temporal_tta(model, uids):
    """Option B: encode each TTA crop with the temporal model, average embeddings."""
    embs = []
    for u in uids:
        if u in tta_temporal_cache:
            crops = tta_temporal_cache[u].float()          # (n_crops, 8, 1280)
            x = crops.to(DEVICE)                           # feed as batch
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                e = model.encode(x).float().cpu()          # (n_crops, D)
            embs.append(e.mean(dim=0))
        else:
            # Fallback: uniform cache
            x = feature_cache[u].float().unsqueeze(0).to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                e = model.encode(x).float().cpu().squeeze(0)
            embs.append(e)
    return torch.stack(embs, 0)


# =============================================================================
# 8. POST-PROCESSING
# =============================================================================

def whiten_fit(X, eps=WHITEN_EPS):
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.T @ Xc) / max(1, Xc.shape[0] - 1)
    d = cov.shape[0]
    cov = cov + eps * torch.eye(d, device=cov.device)
    evals, evecs = torch.linalg.eigh(cov)
    evals = evals.clamp_min(eps)
    order = torch.argsort(evals, descending=True)
    return mu, evecs[:, order], evals[order]


def whiten_apply(X, mu, evecs, evals, dim=PCA_DIM):
    Z = (X - mu) @ evecs
    if dim and 0 < dim < Z.shape[1]:
        Z = Z[:, :dim]; ev = evals[:dim]
    else:
        ev = evals
    return Z / ev.sqrt().unsqueeze(0)


def csls_sim(Q, S, k=CSLS_K):
    if k <= 0: return Q @ S.T
    sim = Q @ S.T
    kq = min(k, sim.shape[1] - 1)
    ks = min(k, sim.shape[0] - 1)
    r_Q = sim.topk(kq, dim=1).values.mean(dim=1, keepdim=True)
    r_S = sim.topk(ks, dim=0).values.mean(dim=0, keepdim=True)
    return sim - 0.5 * (r_Q + r_S)


def retrieval_metrics(sim, q_gloss, p_gloss):
    ranks = torch.argsort(sim, dim=1, descending=True)
    t1 = t5 = t10 = 0; rr = []
    for i in range(sim.shape[0]):
        labs = [p_gloss[j] for j in ranks[i].tolist()]
        tgt = q_gloss[i]
        if labs[0] == tgt: t1 += 1
        if tgt in labs[:5]: t5 += 1
        if tgt in labs[:10]: t10 += 1
        try: r = labs.index(tgt) + 1
        except ValueError: r = len(labs)
        rr.append(1.0 / r)
    n = max(1, sim.shape[0])
    return {"top1": t1/n, "top5": t5/n, "top10": t10/n, "mrr": float(np.mean(rr))}


# =============================================================================
# 9. BUILD EMBEDDINGS
# =============================================================================

print("\n" + "=" * 90)
print("BUILDING EMBEDDINGS")
print("=" * 90)

proto_mp       = build_meanpool_uniform(proto_order)
test_mp        = build_meanpool_uniform(test_order)
proto_mp_tta   = build_meanpool_tta(proto_order)
test_mp_tta    = build_meanpool_tta(test_order)

print(f"mean_pool uniform  : {tuple(proto_mp.shape)}")
print(f"mean_pool TTA      : {tuple(proto_mp_tta.shape)}")

temporal_uniform = {}
temporal_tta     = {}
for name, model in temporal_models.items():
    p_u = build_temporal_uniform(model, proto_order)
    t_u = build_temporal_uniform(model, test_order)
    temporal_uniform[name] = (p_u, t_u)
    print(f"{name:22s} uniform : proto {tuple(p_u.shape)}  test {tuple(t_u.shape)}")

    p_t = build_temporal_tta(model, proto_order)
    t_t = build_temporal_tta(model, test_order)
    temporal_tta[name] = (p_t, t_t)
    print(f"{name:22s} TTA     : proto {tuple(p_t.shape)}  test {tuple(t_t.shape)}")


# =============================================================================
# 10. EVALUATION
# =============================================================================

print("\n" + "=" * 90)
print("EVALUATION")
print("=" * 90)

results = {}

def report(tag, sim):
    m = retrieval_metrics(sim, test_gloss, proto_gloss)
    results[tag] = m
    print(f"  {tag:52s}  Top-1 {m['top1']*100:6.2f}%  Top-5 {m['top5']*100:6.2f}%  "
          f"Top-10 {m['top10']*100:6.2f}%  MRR {m['mrr']*100:6.2f}%")
    return m

print("\n--- Reference baselines (from Stage 7 / 7.1) ---")
report("mean_pool (raw)",
       l2norm(test_mp) @ l2norm(proto_mp).T)

# Whitened mean pool TTA (best Stage 7.1 variant)
X_all_mp = torch.cat([proto_mp_tta, test_mp_tta], 0)
mu_mp, evecs_mp, evals_mp = whiten_fit(X_all_mp)
proto_mp_tta_w = whiten_apply(proto_mp_tta, mu_mp, evecs_mp, evals_mp, dim=PCA_DIM)
test_mp_tta_w  = whiten_apply(test_mp_tta,  mu_mp, evecs_mp, evals_mp, dim=PCA_DIM)
sim_mp_best = l2norm(test_mp_tta_w) @ l2norm(proto_mp_tta_w).T
report("mean_pool + TTA + whiten", sim_mp_best)
report("mean_pool + TTA + whiten + CSLS",
       csls_sim(l2norm(test_mp_tta_w), l2norm(proto_mp_tta_w), k=CSLS_K))


print("\n--- OPTION A: temporal encoders on uniform 8-frame cached features ---")

def evaluate_temporal_path(tag_prefix, proto_e, test_e):
    # Raw
    report(f"{tag_prefix} (raw)",
           l2norm(test_e) @ l2norm(proto_e).T)
    # Whitened + CSLS
    Xt = torch.cat([proto_e, test_e], 0)
    mu_t, evecs_t, evals_t = whiten_fit(Xt)
    dim = min(PCA_DIM, proto_e.shape[1])
    proto_t_w = whiten_apply(proto_e, mu_t, evecs_t, evals_t, dim=dim)
    test_t_w  = whiten_apply(test_e,  mu_t, evecs_t, evals_t, dim=dim)
    report(f"{tag_prefix} + whiten",
           l2norm(test_t_w) @ l2norm(proto_t_w).T)
    report(f"{tag_prefix} + whiten + CSLS",
           csls_sim(l2norm(test_t_w), l2norm(proto_t_w), k=CSLS_K))
    # Concatenation with the whitened mean-pool-TTA
    proto_cat = torch.cat([proto_mp_tta_w, proto_t_w], dim=1)
    test_cat  = torch.cat([test_mp_tta_w,  test_t_w],  dim=1)
    report(f"concat(mean_pool_tta_whiten, {tag_prefix}_whiten)",
           l2norm(test_cat) @ l2norm(proto_cat).T)
    report(f"concat(mean_pool_tta_whiten, {tag_prefix}_whiten) + CSLS",
           csls_sim(l2norm(test_cat), l2norm(proto_cat), k=CSLS_K))


for name, (p_u, t_u) in temporal_uniform.items():
    evaluate_temporal_path(f"{name} (uniform)", p_u, t_u)


print("\n--- OPTION B: temporal encoders on real per-crop TTA ---")

for name, (p_t, t_t) in temporal_tta.items():
    evaluate_temporal_path(f"{name} (TTA)", p_t, t_t)


# =============================================================================
# 11. SUMMARY AND SAVE
# =============================================================================

print("\n" + "=" * 90)
print("SUMMARY — ALL VARIANTS")
print("=" * 90)

rows = [{"variant": k, "top1": v["top1"], "top5": v["top5"],
         "top10": v["top10"], "mrr": v["mrr"]} for k, v in results.items()]
df = pd.DataFrame(rows).sort_values("top1", ascending=False).reset_index(drop=True)
print(df.to_string(index=False))
df.to_csv(OUT_DIR / "stage7_2_results.csv", index=False)

best = df.iloc[0]
raw_mp = results["mean_pool (raw)"]
print(f"\nBest variant: {best['variant']}")
print(f"  Top-1 {best['top1']*100:.2f}%   Top-5 {best['top5']*100:.2f}%   "
      f"Top-10 {best['top10']*100:.2f}%   MRR {best['mrr']*100:.2f}%")
print(f"\nImprovement over raw mean_pool:")
print(f"  Top-1 : {(best['top1']-raw_mp['top1'])*100:+.2f} pp  "
      f"(relative {best['top1']/max(1e-9,raw_mp['top1']):.2f}x)")

config = {
    "stage": "7.2",
    "seed": SEED,
    "n_tta_crops": N_TTA_CROPS,
    "pca_dim": PCA_DIM,
    "csls_k": CSLS_K,
    "whiten_eps": WHITEN_EPS,
    "option_a": "temporal encoders on uniform 8-frame cached features",
    "option_b": "temporal encoders on real per-crop TTA (n_crops x 8 x 1280)",
    "best_variant": best["variant"],
    "best_top1": float(best["top1"]),
    "note": ("No training. Whitening fit on joint prototype+test feature cloud "
             "without labels. CSLS rank-based. TTA deterministic. No test label "
             "used for any choice."),
}
with open(OUT_DIR / "stage7_2_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\nArtifacts written to:\n    {OUT_DIR}")
print("\n" + "=" * 90)
print("STAGE 7.2 COMPLETE")
print("=" * 90)

STAGE 7.2 — CORRECTED TEMPORAL TTA + PATH 2 CONCATENATION
Device            : cuda
GPU               : NVIDIA GeForce RTX 4050 Laptop GPU
Output directory  : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage7_2_corrected_tta
TTA crops         : 3
PCA dim           : 512
CSLS k            : 10
Prototype (valid): 4,763
Test (valid)     : 2,285

LOADING FEATURE CACHES
Uniform cache  : all_video_features_75f7fb9f.pt
  videos       : 7,048
TTA mean cache : tta3_features.pt
  videos       : 7,048

TEMPORAL TTA CACHE (Option B)
Extracting full temporal TTA features (one-time, ~15 min on RTX 4050)
Each video is decoded once and 3 temporal windows are sampled from it,
so per-video decode cost is amortized across crops.
Videos to process: 7,048
  [  200/ 7048]  14.4 vid/s  ETA 7.9 min
  [  400/ 7048]  14.8 vid/s  ETA 7.5 min
  [  600/ 7048]  14.9 vid/s  ETA 7.2 min
  [  800/ 7048]  14.9 vid/s  ETA 7.0 min
  [ 1000/ 7048]  15.0 vid/s  ETA 6.7 min
  [ 1200/ 7048]  14.8 vid/s  ETA 6.6 min
  [ 14